# Intelligent Hospital Workflow Assistant
## Deep Learning Project — Milestone 2



---
## Setup

In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)

Project root: /Users/ramezhany/Uni/Sem 10/Deep Learning/Project/DLCV-Milestone 2


## Imports

In [2]:
from dotenv import load_dotenv
import os
from backend.database import engine
from backend.models import Base
from backend.seed_data import seed
from backend.database import get_session
from backend.models import Doctor
from backend.models import User
from backend.tools import (
    get_patient_history,
    find_available_doctor,
    book_appointment,
    save_medical_report,
    send_confirmation_email,
)
import json
from backend.agent import build_agent_executor
from backend.tools import get_doctor_slots, get_all_appointments

load_dotenv()

assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set — edit .env first'
print('API key loaded.')

API key loaded.


In [3]:


Base.metadata.create_all(bind=engine)
seed()

Database already seeded. Skipping.


In [4]:


db = get_session()
doctors = db.query(Doctor).all()
db.close()

print(f'{len(doctors)} doctors seeded:')
for d in doctors:
    print(f'  [{d.id}] {d.name} — {d.specialty} ({d.department})')

8 doctors seeded:
  [1] Dr. Sarah Mitchell — cardiology (Cardiology)
  [2] Dr. James Okafor — neurology (Neurology)
  [3] Dr. Lena Hoffmann — general (General Practice)
  [4] Dr. Ahmed Al-Rashid — orthopedics (Orthopedics)
  [5] Dr. Priya Sharma — dermatology (Dermatology)
  [6] Dr. Carlos Mendes — gastroenterology (Gastroenterology)
  [7] Dr. Fatima Zahra — pulmonology (Pulmonology)
  [8] Dr. Wei Chen — endocrinology (Endocrinology)


In [5]:


db = get_session()
for email, age, history in [
    ('ramezhany986@gmail.com', 45, 'Type 2 diabetes, hypertension'),
    ('mohamedshady2003@gmail.com', 28, 'None'),
]:
    if not db.query(User).filter(User.email == email).first():
        db.add(User(email=email, age=age, chronic_disease_history=history))
db.commit()
db.close()
print('Test patients registered.')

Test patients registered.



##  Tool Demos

In [6]:


def pretty(result):
    print(json.dumps(result, indent=2, default=str))

In [7]:
print('=== Tool 1: get_patient_history ===')
pretty(get_patient_history.invoke({'email': 'ramezhany986@gmail.com'}))

=== Tool 1: get_patient_history ===
{
  "success": true,
  "data": {
    "email": "ramezhany986@gmail.com",
    "age": 45,
    "chronic_disease_history": "Type 2 diabetes, hypertension",
    "past_appointments": [
      {
        "appointment_id": 3,
        "doctor": "Dr. Sarah Mitchell",
        "specialty": "cardiology",
        "slot": "2026-04-25 09:00",
        "priority": 0,
        "report_summary": "URGENT - Priority 0 Triage Assessment. Patient: Male/Female, 45 years old. Known history of Type 2 diabetes and hypertension \u2014 both significant cardiovascular risk factors. Chief Complaint: Severe chest pain radiating to the left arm, accompanied by difficulty breathing. Onset: Approximately 20 minutes prior to intake. Symptom profile is consistent with a potential acute myocardial infarction (AMI) or other serious acute coronary syndrome. Immediate cardiology evaluation is required. Patient was triaged as Priority 0 (URGENT) and routed to Cardiology without delay. NOTE: This 

In [9]:
print('=== Tool 2: find_available_doctor (cardiology) ===')
doctor_result = find_available_doctor.invoke({'specialty': 'Dermatology'})
pretty(doctor_result)

=== Tool 2: find_available_doctor (cardiology) ===
{
  "success": true,
  "data": {
    "doctor_id": 5,
    "doctor_name": "Dr. Priya Sharma",
    "specialty": "dermatology",
    "department": "Dermatology",
    "slot_id": 41,
    "slot_datetime": "2026-04-25 09:00"
  }
}


In [10]:
print('=== Tool 3: book_appointment ===')
d = doctor_result['data']
booking = book_appointment.invoke({
    'user_email': 'alice@example.com',
    'doctor_id': d['doctor_id'],
    'slot_id': d['slot_id'],
    'priority': 1,
})
pretty(booking)

=== Tool 3: book_appointment ===
{
  "success": true,
  "data": {
    "appointment_id": 1,
    "doctor_name": "Dr. Priya Sharma",
    "specialty": "dermatology",
    "slot_datetime": "2026-04-25 09:00",
    "priority": 1
  }
}


In [11]:
print('=== Tool 4: save_medical_report ===')
appt_id = booking['data']['appointment_id']
pretty(save_medical_report.invoke({
    'appointment_id': appt_id,
    'summary': 'Patient reports persistent chest tightness for 3 days. Known hypertension. Triage: Priority 1.',
    'medication_recommendations': 'Continue antihypertensive medication. Avoid strenuous activity.',
}))

=== Tool 4: save_medical_report ===
{
  "success": true,
  "data": {
    "report_id": 1,
    "appointment_id": 1
  }
}


In [13]:
print('=== Tool 5: send_confirmation_email (logged, no SMTP configured) ===')
pretty(send_confirmation_email.invoke({
    'to_email': 'ramezhany986@gmail.com',
    'doctor_name': d['doctor_name'],
    'slot_datetime': d['slot_datetime'],
    'department': d['department'],
}))

=== Tool 5: send_confirmation_email (logged, no SMTP configured) ===
{
  "success": true,
  "data": {
    "sent_to": "ramezhany986@gmail.com",
    "method": "smtp"
  }
}



## Agent Demo

In [9]:


executor = build_agent_executor()

/Users/ramezhany/Uni/Sem 10/Deep Learning/Project/DLCV-Milestone 2/backend/agent.py:97: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = session_memory or ConversationBufferMemory(


In [10]:
print('='*60)
print('SCENARIO 1: Urgent — chest pain + shortness of breath (Priority 0)')
print('='*60)

r = executor.invoke({
    'input': 'Patient email: ramezhany986@gmail.com\n\nPatient message: I have severe chest pain radiating to my left arm and I can barely breathe. It started 20 minutes ago.'
})
print('\n--- AGENT FINAL RESPONSE ---')
print(r['output'])

SCENARIO 1: Urgent — chest pain + shortness of breath (Priority 0)


> Entering new AgentExecutor chain...
Thought: This is a Priority 0 URGENT case. The patient is describing classic heart attack symptoms - severe chest pain radiating to the left arm combined with difficulty breathing, starting 20 minutes ago. I must act immediately without asking follow-up questions. Let me start by retrieving the patient's history.

Action: get_patient_history
Action Input: ramezhany986@gmail.com{'success': True, 'data': {'email': 'ramezhany986@gmail.com', 'age': 45, 'chronic_disease_history': 'Type 2 diabetes, hypertension', 'past_appointments': [{'appointment_id': 3, 'doctor': 'Dr. Sarah Mitchell', 'specialty': 'cardiology', 'slot': '2026-04-25 09:00', 'priority': 0, 'report_summary': 'URGENT - Priority 0 Triage Assessment. Patient: Male/Female, 45 years old. Known history of Type 2 diabetes and hypertension — both significant cardiovascular risk factors. Chief Complaint: Severe chest pain radiati

In [ ]:
print('='*60)
print('SCENARIO 2: Multi-turn — vague symptoms requiring follow-up')
print('='*60)

executor2 = build_agent_executor()

r1 = executor2.invoke({'input': 'Patient email: alice@example.com\n\nPatient message: I have been feeling dizzy and tired lately.'})
print('[AGENT TURN 1]:', r1['output'])

r2 = executor2.invoke({'input': 'Patient email: alice@example.com\n\nPatient message: About 5 days. The dizziness is worst in the morning and I also have a headache.'})
print('\n[AGENT TURN 2]:', r2['output'])

In [ ]:
print('='*60)
print('SCENARIO 3: Routine — prescription renewal (Priority 3)')
print('='*60)

executor3 = build_agent_executor()
r = executor3.invoke({'input': 'Patient email: bob@example.com\n\nPatient message: I need to renew my asthma inhaler prescription. No new symptoms, just running out.'})
print('[AGENT RESPONSE]:', r['output'])

In [ ]:
get_doctor_slots.invoke({"doctor_id": "1"})

{'success': True,
 'data': {'doctor_id': 1,
  'doctor_name': 'Dr. Sarah Mitchell',
  'specialty': 'cardiology',
  'slots': [{'slot_id': 1, 'datetime': '2026-04-25 09:00', 'status': 'booked'},
   {'slot_id': 2, 'datetime': '2026-04-25 11:00', 'status': 'available'},
   {'slot_id': 3, 'datetime': '2026-04-26 09:00', 'status': 'available'},
   {'slot_id': 4, 'datetime': '2026-04-26 11:00', 'status': 'available'},
   {'slot_id': 5, 'datetime': '2026-04-27 09:00', 'status': 'available'},
   {'slot_id': 6, 'datetime': '2026-04-27 11:00', 'status': 'available'},
   {'slot_id': 7, 'datetime': '2026-04-28 09:00', 'status': 'available'},
   {'slot_id': 8, 'datetime': '2026-04-28 11:00', 'status': 'available'},
   {'slot_id': 9, 'datetime': '2026-04-29 09:00', 'status': 'available'},
   {'slot_id': 10, 'datetime': '2026-04-29 11:00', 'status': 'available'}]}}

In [4]:
get_all_appointments.invoke({"_": ""})

{'success': True,
 'data': [{'appointment_id': 1,
   'patient_email': 'alice@example.com',
   'doctor': 'Dr. Priya Sharma',
   'specialty': 'dermatology',
   'slot_datetime': '2026-04-25 09:00',
   'priority': 1,
   'has_report': True},
  {'appointment_id': 2,
   'patient_email': 'bob@example.com',
   'doctor': 'Dr. James Okafor',
   'specialty': 'neurology',
   'slot_datetime': '2026-04-25 09:00',
   'priority': 0,
   'has_report': False},
  {'appointment_id': 3,
   'patient_email': 'ramezhany986@gmail.com',
   'doctor': 'Dr. Sarah Mitchell',
   'specialty': 'cardiology',
   'slot_datetime': '2026-04-25 09:00',
   'priority': 0,
   'has_report': True}]}

In [1]:
from backend.tools import *
get_medical_report.invoke({"appointment_id": "3"})

{'success': True,
 'data': {'report_id': 2,
  'appointment_id': 3,
  'summary': 'URGENT - Priority 0 Triage Assessment. Patient: Male/Female, 45 years old. Known history of Type 2 diabetes and hypertension — both significant cardiovascular risk factors. Chief Complaint: Severe chest pain radiating to the left arm, accompanied by difficulty breathing. Onset: Approximately 20 minutes prior to intake. Symptom profile is consistent with a potential acute myocardial infarction (AMI) or other serious acute coronary syndrome. Immediate cardiology evaluation is required. Patient was triaged as Priority 0 (URGENT) and routed to Cardiology without delay. NOTE: This is a preliminary AI triage assessment and is subject to physician review and clinical judgment.',
  'medication_recommendations': 'No medications recommended at this stage. Await physician evaluation. If confirmed AMI protocol: aspirin administration and emergency intervention per cardiology team discretion.'}}

---
## Phase 4 — FastAPI Backend

In [ ]:
import subprocess, time

server = subprocess.Popen(
    ['python3', '-m', 'uvicorn', 'backend.main:app', '--host', '127.0.0.1', '--port', '8000'],
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print('Server started. PID:', server.pid)

In [ ]:
import requests
BASE = 'http://127.0.0.1:8000'

print('Health:', requests.get(f'{BASE}/health').json())
print('Identify alice:', requests.post(f'{BASE}/identify', json={'email': 'alice@example.com'}).json())
print('Identify unknown:', requests.post(f'{BASE}/identify', json={'email': 'new@example.com'}).json())

In [ ]:
resp = requests.post(f'{BASE}/register', json={
    'email': 'new@example.com',
    'age': 35,
    'chronic_disease_history': 'Asthma',
})
print('Register:', resp.json())

In [ ]:
resp = requests.post(f'{BASE}/chat', json={
    'email': 'new@example.com',
    'message': 'I have had a skin rash on my arm for a week. It is itchy and slightly red.',
})
data = resp.json()
print('Session ID:', data['session_id'])
print('\nAgent Response:\n', data['response'])

In [ ]:
server.terminate()
print('Server stopped.')

---
## Phase 5 — Evaluation

Priority assignment accuracy across labelled test cases.

In [ ]:
import re

TEST_CASES = [
    ('Severe chest pain radiating to left arm, started 10 minutes ago', 0),
    ('Sudden loss of speech and facial drooping on the right side', 0),
    ('High fever 39.5 degrees for two days with severe headache', 1),
    ('Moderate knee pain after a fall, can still walk', 2),
    ('Dry cough and mild fatigue for the past week', 2),
    ('Routine blood pressure check, no symptoms', 3),
    ('Prescription renewal for daily metformin', 3),
]

results = []
for desc, expected in TEST_CASES:
    ex = build_agent_executor()
    resp = ex.invoke({'input': f'Patient email: bob@example.com\n\nPatient message: {desc}'})
    output = resp['output']
    match = re.search(r'[Pp]riority\s*[:\-]?\s*(\d)', output)
    predicted = int(match.group(1)) if match else -1
    correct = (predicted == expected)
    results.append((desc[:55], expected, predicted, correct))
    print(f'[{"PASS" if correct else "FAIL"}] Expected P{expected}, Got P{predicted} | {desc[:55]}')

accuracy = sum(r[3] for r in results) / len(results)
print(f'\nAccuracy: {accuracy:.0%} ({sum(r[3] for r in results)}/{len(results)})')

---
## Summary

| Component | Status |
|---|---|
| PostgreSQL database (6 tables) | Done |
| SQLAlchemy ORM models | Done |
| Doctor + slot seeding (8 doctors, 10 slots each) | Done |
| `get_patient_history` tool | Done |
| `find_available_doctor` tool | Done |
| `book_appointment` tool (atomic, no double-booking) | Done |
| `save_medical_report` tool | Done |
| `send_confirmation_email` tool | Done |
| LangChain ReAct agent (Claude claude-sonnet-4-6) | Done |
| FastAPI backend (4 endpoints) | Done |
| Multi-turn conversation memory | Done |
| Conversation audit trail in DB | Done |

All agent outputs are framed as preliminary recommendations pending physician review.